# Rebuild Results CSV From Saved Checkpoints

This notebook loads existing `.pth` models, re-evaluates them, and writes a fresh CSV without overwriting previous results.


## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd
import torch
from datetime import datetime

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
SCRIPT_DIR = PROJECT_ROOT / 'baseline_models_scripts'
MODEL_DIR = SCRIPT_DIR / 'runs' / 'models'
OUTPUT_DIR = SCRIPT_DIR / 'runs'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
PROJECT_ROOT, MODEL_DIR, OUTPUT_DIR, device

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/models'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 device(type='cuda', index=0))

## 2. Import Training Utilities

In [2]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from baseline_models_scripts.train_ttd import (
    EXPERIMENTS,
    baseline_model_pipeline,
    build_model,
    safe_name,
    TTDCombinedLoss,
)
from utils import val_loop


## 3. Experiment Selection

Choose the experiments, architecture, and backbone to evaluate.

In [3]:
SETTINGS = [
    'Single-TB',
    'Shift-TA_TC-to-TB_10pct',
    'Shift-TA_TB-to-TC_10pct',
]

ARCH = 'UnetPlusPlus'         # change to Unet / UnetPlusPlus / FPN / DeepLabV3Plus / Segformer as needed
ENCODER = 'resnet34'  # change to resnet34 / resnet50 / efficientnet-b0 / mit_b0 as needed
ENCODER_WEIGHTS = 'imagenet'

BATCH_SIZE = 16
IMG_SIZE = 512
CRACK_WEIGHT = 20.0
W_CE = 0.5
W_DICE = 0.5


## 4. Evaluate Checkpoints and Build New Results CSV

In [4]:
rows = []

for setting in SETTINGS:
    exp_cfg = EXPERIMENTS[setting]
    _, val_dl, test_dl, custom_stats, _ = baseline_model_pipeline(
        base_dir=PROJECT_ROOT,
        dict_files=exp_cfg,
        bs=BATCH_SIZE,
        img_size=IMG_SIZE,
    )

    model = build_model(
        arch=ARCH,
        encoder_name=ENCODER,
        encoder_weights=ENCODER_WEIGHTS,
        classes=2,
    )

    model_name = safe_name(f'{ARCH}-{ENCODER}-{ENCODER_WEIGHTS}_{setting}')
    ckpt_path = MODEL_DIR / f'{model_name}.pth'
    if not ckpt_path.exists():
        print(f'Skip (missing checkpoint): {ckpt_path}')
        continue

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)

    weights = torch.tensor([1.0, CRACK_WEIGHT], device=device)
    loss_fn = TTDCombinedLoss(ce_weight_tensor=weights, w_ce=W_CE, w_dice=W_DICE)

    v_loss, v_iou, v_f1, v_recall, v_prec = val_loop(model, device, val_dl, loss_fn, is_test=True)
    t_loss, t_iou, t_f1, t_recall, t_prec = val_loop(model, device, test_dl, loss_fn, is_test=True)

    rows.append({
        'Experiment': model_name,
        'Val_Loss': v_loss,
        'Val_IoU': v_iou,
        'Val_F1': v_f1,
        'Val_Recall': v_recall,
        'Val_Prec': v_prec,
        'Test_Loss': t_loss,
        'Test_IoU': t_iou,
        'Test_F1': t_f1,
        'Test_Recall': t_recall,
        'Test_Prec': t_prec,
    })

results_df = pd.DataFrame(rows)
results_df

Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 424/424 [00:00<00:00, 1791.61it/s]


Sanitizing test masks...


Sanitizing Masks: 100%|██████████| 48/48 [00:00<00:00, 1766.50it/s]


Generating Dataloaders...

Pipeline Ready:
 - Training batches: 20
 - Validation batches: 6
 - Testing batches: 3
Loading CSV metadata...
Sanitizing training and validation masks...


...

Sanitizing Masks: 100%|██████████| 1028/1028 [00:00<00:00, 22012.96it/s]


Sanitizing test masks...


Sanitizing Masks: 100%|██████████| 38/38 [00:00<00:00, 1626.78it/s]

Generating Dataloaders...

Pipeline Ready:
 - Training batches: 50
 - Validation batches: 15
 - Testing batches: 3


,Experiment,Val_Loss,Val_IoU,Val_F1,Val_Recall,Val_Prec,Test_Loss,Test_IoU,Test_F1,Test_Recall,Test_Prec
0,UnetPlusPlus-resnet34-imagenet_Single-TB,5.708144,0.450570,0.611742,0.706225,0.566277,6.303437,0.319969,0.484504,0.433426,0.555844
1,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-TB_10pct,5.604758,0.565192,0.701186,0.742247,0.684563,6.531316,0.308235,0.470894,0.571863,0.419408
2,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,5.446961,0.592520,0.735340,0.764691,0.722272,5.708749,0.298909,0.453146,0.442893,0.482008


## 5. Save Results Without Overwriting

In [5]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
# output_name = f'{ARCH}_{ENCODER}_{ENCODER_WEIGHTS}.csv'
output_name = f'layer3_arch_{ARCH}_{ENCODER}.csv'
output_path = OUTPUT_DIR / output_name
results_df.to_csv(output_path, index=False)
output_path

Path('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer3_arch_UnetPlusPlus_resnet34.csv')